In [ ]:
!python --version

In [ ]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


### Defaults

In [ ]:
ROOT = Path().resolve().parent
root0 = ROOT / "data"

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


In [ ]:
np.array(prog_list)

### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
prog_id = 'TCGA'
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=force, verbose=verbose)

primary_sites = np.unique(df_psi.primary_site)
primary_sites[:10]

In [ ]:
primary_site = 'Breast'
gdc.set_primary_site(primary_site = primary_site)

primary_site = gdc.primary_site
psi_id = gdc.psi_id

psi_id, primary_site

In [ ]:
verbose=False

df_cases, df_all_samples, df_all_mut, all_barcode_list = gdc.get_filtered_tables(primary_site=primary_site, verbose=verbose)

In [ ]:
df_cases.head(2)

In [ ]:
case_id_list = np.unique(df_cases.case_id)
print(len(case_id_list))

case_id_list[:3]

In [ ]:
row = df_cases.iloc[0]

subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue

subtype_global, tumor_class, subtype_global

In [ ]:
df_cases.head(3)

### Preparing to get VCF files from TCGA programa

In [ ]:
case_id_list = np.unique(df_cases.case_id)
print(len(case_id_list))

row = df_cases.iloc[0]

subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue

print(subtype_global, tumor_class, subtype_global)

In [ ]:
verbose=True
force=True

df_vcf = gdc.get_VCF_files(subtype_global=subtype_global, tumor_class=tumor_class, subtype_tissue=subtype_tissue, 
                       batch_cases=20, batch_size=20, timeout=120, force=force, verbose=verbose)

print(len(df_vcf))
                    

In [ ]:
df_vcf.head(3).T

### Development & tests

In [ ]:
def get_VCF_files(subtype_global:str, tumor_class:str, subtype_tissue:str, 
			    batch_cases:int=50, batch_size:int=200, timeout:int=100,
			    force:bool=False, verbose:bool=False) -> pd.DataFrame:

    df_vcf = pd.DataFrame()
    gdc.df_vcf = df_vcf

    gdc.set_s_case(subtype_global, tumor_class, subtype_tissue)

    df_cases, _, _ = gdc.get_cases_and_subtypes(batch_size=batch_size, do_filter=False, 
                                                force=False, verbose=verbose)

    gdc.df_cases = df_cases

    if df_cases is None or df_cases.empty:
        print(f"No cases found while searching for '{gdc.s_case}'")
        return df_vcf
          
    case_id_list = list(df_cases.case_id)
    case_id_list.sort()

    N_cases = len(case_id_list)
    print(f">>> {N_cases} cases")

    gdc.fname_vcf_files0 = 'vcf_files_for_%s.tsv'
    fname = gdc.fname_vcf_files0%(gdc.s_case)
    fname = title_replace(fname)
    filename = os.path.join(gdc.root_psi, fname)

    #-------------------------- batch loop ---------------------------
    all_hits = []
    from_ = 0

    size_ = batch_size
    total = None

    # print("Searching: ", end='')

    ini = -batch_cases
    end = 0

    while(True):
        ini += batch_cases
        end += batch_cases

        if ini >= N_cases:
            break
        
        if end > N_cases:
            end = N_cases

        print(f"{ini}-{end} ", end='')

        lista = case_id_list[ini: end]

        print('>>> 0a')
        filters = {
            "op": "and",
            "content": [
                {"op": "in",
                    "content": {"field": "cases.case_id", "value": lista}
                },
                {"op": "=",
                    "content": {"field": "files.data_format", "value": "VCF"}
                },
                {"op": "in",
                    "content": {"field": "files.data_type",
                                "value": [
                                    "Raw Simple Somatic Mutation",
                                    "Masked Somatic Mutation"
                                ]
                    }
                }
            ]
        }
    
        try:
            while True:
                print(".", end='')

                params = {
                    "filters": json.dumps(filters),
                    "fields": ",".join([
                        "file_id",
                        "file_name",
                        "data_type",
                        "data_format",
                        "analysis.workflow_type",
                        "cases.case_id",
                        "cases.submitter_id",
                        "cases.samples.sample_id",
                        "cases.samples.submitter_id",
                        "cases.samples.sample_type",
                    ]),
                    "format": "JSON",
                    "size": size_,
                    "from": from_
                }
                
                res = requests.get(gdc.url_gdc_files, params=params, timeout=timeout)
                """
                res.raise_for_status()   # raises if HTTP 4xx/5xx
                print("status:", res.status_code)
                print("content-type:", res.headers.get("Content-Type"))
                print("text:", res.text[:500])   # inspect response before parsing
                """
                response = res.json()

                if 'data' not in response.keys():
                    print(f"No data found while searching for '{gdc.psi_id}' cases {case_id_list}")
                    print(">>> response", response)
                    return df_vcf
            
                hits = response.get("data", {}).get("hits", [])

                if total is None:
                    total = response["data"]["pagination"]["total"]
                
                if not hits:
                    break

                all_hits.extend(hits)
                from_ += size_

            print("\n")

            if all_hits == []:
                print(f"No files were found for {psi_id} cases {case_id_list}")
                return df_vcf

            #------------ lost data? ------------------
            N = len(all_hits)

            if N < total:
                print(f"⚠️ Warning: results truncated — consider pagination - all hits = {N};  Total paginated {total} ")
            else:
                if verbose: print(f"👉 Returned {N} / Total paginated {total}")

            #------------ having all hits -------------
            records = []
            print('>>> 2')

            for hit in all_hits:
                for case in hit.get("cases", []):
                    for sample in case.get("samples", []):
                        records.append({
                            "case_id": case["case_id"],
                            "submitter_id": case["submitter_id"],
                            "sample_id":   sample["sample_id"],
                            "sample_type": sample["sample_type"],
                            "barcode_sample":  sample["submitter_id"],
                            "file_id":    hit["file_id"],
                            "file_name":  hit["file_name"],
                            "data_type":  hit["data_type"],
                            "data_format": hit["data_format"],
                            "workflow_type": hit["analysis"]["workflow_type"],
                        })
                        print('>>> 3')

            df_vcf = pd.DataFrame(records)
            cols = list(df_vcf.columns)

            # 🔹 Metadata 
            df_vcf['psi_id'] = psi_id
            df_vcf['subtype_global'] = subtype_global
            df_vcf['tumor_class'] = tumor_class
            df_vcf['subtype_tissue'] = subtype_tissue

            cols = ["psi_id", "subtype_global", "tumor_class", "subtype_tissue"] + cols

            df_vcf = df_vcf.sort_values(["case_id", "sample_type"], ascending=[False,False]).reset_index(drop=True)
            df_vcf.reset_index(drop=True, inplace=True)

            _ = pdwritecsv(df_vcf, fname, gdc.root_psi, verbose=verbose)

        except Exception as e:
            print(f"Error for searching files for {gdc.s_case}'. error: {e}")
            print("Trye to diminish the batch size - like ~20")
            return df_vcf

    gdc.df_vcf = df_vcf

    return df_vcf

In [ ]:
run_here = True

hits = []

if run_here:
    lista = list(case_id_list[:3])

    filters = {
        "op": "and",
        "content": [
            {"op": "in",
                "content": {"field": "cases.case_id", "value": lista}
            },
            {"op": "=",
                "content": {"field": "files.data_format", "value": "VCF"}
            },
            {"op": "in",
                "content": {"field": "files.data_type",
                            "value": [
                                "Raw Simple Somatic Mutation",
                                "Masked Somatic Mutation"
                            ]
                }
            }
        ]
    }

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join([
            "file_id",
            "file_name",
            "data_type",
            "data_format",
            "analysis.workflow_type",
            "cases.case_id",
            "cases.submitter_id",
            "cases.samples.sample_id",
            "cases.samples.submitter_id",
            "cases.samples.sample_type",
        ]),
        "format": "JSON",
        "size": 2000
    }

    res = requests.get(gdc.url_gdc_files, params=params)

    response = res.json()

    if 'data' not in response.keys():
        print(f"No data found while searching for '{gdc.psi_id}' cases {case_id_list}")
        print(">>> response", response)
        hits = []
    else:
        hits = response.get("data", {}).get("hits", [])

    print(len(hits))
    hits[:3]

if hits:

    records = []

    for hit in hits:
        for case in hit.get("cases", []):
            for sample in case.get("samples", []):
                records.append({
                    "case_id": case["case_id"],
                    "submitter_id": case["submitter_id"],
                    "sample_id":   sample["sample_id"],
                    "sample_type": sample["sample_type"],
                    "barcode_sample":  sample["submitter_id"],
                    "file_id":    hit["file_id"],
                    "file_name":  hit["file_name"],
                    "data_type":  hit["data_type"],
                    "data_format": hit["data_format"],
                    "workflow_type": hit["analysis"]["workflow_type"],
                })

    df_vcf = pd.DataFrame(records)
    cols = list(df_vcf.columns)

    # 🔹 Metadata 
    df_vcf['psi_id'] = psi_id
    df_vcf['subtype_global'] = subtype_global
    df_vcf['tumor_class'] = tumor_class
    df_vcf['subtype_tissue'] = subtype_tissue
    # df_vcf['stage'] = gdc.get_stage_from_cases(df_vcf.case_id.tolist())

    cols = ["psi_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"] + cols

    df_vcf = df_vcf.sort_values(["case_id", "sample_type"], ascending=[False,False]).reset_index(drop=True)
    df_vcf.reset_index(drop=True, inplace=True)



